## Exporting Data for a Single Organization ID and all Meter Point ID's

### Notebook Structure

1. **Imports & Definitions**  
    Initialization of required libraries, configuration settings, and helper functions for database access and file handling.
    
2. **Exporting Data From Database**  
    Retrieval of all relevant data for the specified **OrgID** and **mp_id** and prepare the date
    
3. **Save as CSV**  
    All retrieved datasets are exported to CSV files.

### 01 Imports & Definitions

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from datetime import datetime
from modules.data_loading import load_energy_community_data, load_all_metering_points_in_energy_community_data, get_postgres_engine
from modules.load_config import load_params_from_yaml


In [ ]:
org_ids = [ 37] # 17 # 13 # 2 # 1 # 13, 17
time_start = datetime(2025, 1, 1)
time_end = datetime(2025, 9, 30)

date = datetime.now()   # aktuelles Datum
path_to_configs = "../../config/"

notebooks_config_file = "notebooks.yaml"
config_dict = load_params_from_yaml([f"{path_to_configs}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]


In [ ]:

descriptions = {
    "Data extraction date": date,
    "org_id": "Organisation ID",
    "time": "Measurement timestamp",
    "metering_points_cnt": "Number of metering points in the REC",
    "consumer_count": "Number of consumers in the REC",
    "generator_count": "Number of producers in the REC",
    "sum_wt_meas_cons": "Sum of measured consumption weighted by participation factor",
    "sum_comm_pot": "Sum of community potential",
    "sum_comm_cov": "Sum of community coverage",
    "sum_wt_meas_gen": "Sum of measured generation weighted by participation factor",
    "sum_wt_surp_gen": "Sum of residual surplus weighted by participation factor"
}

### 02 Exporting Data From Database

In [ ]:
load_dotenv()
ssh_host = os.getenv("SSH_HOST")
ssh_port = int(os.getenv("SSH_PORT"))
ssh_user = os.getenv("SSH_USER")
ssh_pw = os.getenv("SSH_PASSWORD")

postgres_server_ip = os.getenv("POSTGRES_SERVER_IP")
postgres_port = int(os.getenv("POSTGRES_PORT"))

In [ ]:
with SSHTunnelForwarder(
    (ssh_host, ssh_port),
    ssh_username=ssh_user,
    ssh_password=ssh_pw,
    remote_bind_address=(ssh_host, postgres_port),
    local_bind_address=(postgres_server_ip, postgres_port)
) as tunnel:
    for act_org_id in org_ids:
        df_mps = load_all_metering_points_in_energy_community_data(
            org_id=act_org_id,
            time_start=time_start,
            time_end=time_end,
            sql_engine=get_postgres_engine(),
        )
        output_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=act_org_id)
        df_mps.to_csv(f"{path_to_local_data}{output_filename}", index=False)
        print(f"saved file to: {path_to_local_data}{output_filename}")
        
        # DataFrame with columns
        df = pd.DataFrame(columns=descriptions.keys())

        # Create CSV
        meta_df = pd.DataFrame({
            "column_name": df.columns,
            "description": df.columns.map(descriptions)
        })

        meta_df.to_csv(f"{path_to_local_data}{output_filename}_metadata.csv", index=False)